# Blind Image Deconvolution — Real Image Enhancement
### MATH 441 Convex Optimization

Old Optimization Function:
$$
\min_{k,\,\mathbf{u}}\;
\|k*\mathbf{u}-\mathbf{f}\|_2^2
+\lambda_1\underbrace{\|\nabla\mathbf{u}\|_1}_{\text{TV}}
+\lambda_2\underbrace{\|D\mathbf{u}\|_2^2}_{\text{Tikhonov}}
+\mu\underbrace{\|k\|_1}_{\text{kernel sparsity}}
\quad\text{s.t.}\quad k\geq0,\;\mathbf{1}^Tk=1,\;0\leq\mathbf{u}\leq255
$$

New Optimization Function:
$$
\min_{\mathbf{u}}\;
\|k*\mathbf{u}-\mathbf{f}\|_2^2
+\lambda_1\underbrace{\|\nabla\mathbf{u}\|_1}_{\text{TV}}
\quad\text{s.t.}\quad k\geq0,\;\mathbf{1}^Tk=1,\;0\leq\mathbf{u}\leq 1\; {\text{assuming normalized image values}}
$$


In [1]:
pip install cvxpy

Note: you may need to restart the kernel to use updated packages.


## Importing Packages

In [2]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.signal import convolve2d

## Defining Functions

In [3]:
def get_gaussian_kernel(size=5, sigma=1.0):
    """Generates a 2D Gaussian kernel."""
    ax = np.linspace(-(size - 1) / 2., (size - 1) / 2., size)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-0.5 * (np.square(xx) + np.square(yy)) / np.square(sigma))
    return kernel / np.sum(kernel)

def create_sparse_blur_matrix(img_shape, kernel):
    """
    Constructs a sparse matrix K that performs 2D convolution.
    When K @ image.flatten() is computed, it's equivalent to convolve2d(image, kernel).
    """
    M, N = img_shape
    kH, kW = kernel.shape
    pad_h, pad_w = kH // 2, kW // 2

    rows, cols, vals = [], [], []
    
    # Loop over every pixel in the image
    for i in range(M):
        for j in range(N):
            row_idx = i * N + j
            # Loop over the kernel
            for ki in range(kH):
                for kj in range(kW):
                    ii = i + ki - pad_h
                    jj = j + kj - pad_w
                    # Check boundary bounds (Zero padding assumption)
                    if 0 <= ii < M and 0 <= jj < N:
                        col_idx = ii * N + jj
                        rows.append(row_idx)
                        cols.append(col_idx)
                        vals.append(kernel[ki, kj])
                        
    return sp.csr_matrix((vals, (rows, cols)), shape=(M*N, M*N))

## Importing Image and Resizing to 128x128

In [4]:
from PIL import Image
import os
image_path = "kitten.jpg"

img_size = 128

if not os.path.exists(image_path):
    print(f"Error: {image_path} not found.")
    print("Please ensure your image is in the same folder as this script.")
    # Fallback to a simple gradient if file is missing so code doesn't crash
    f_img = np.outer(np.linspace(0, 1, img_size), np.linspace(0, 1, img_size))
else:
    # Open, convert to grayscale, resize, and normalize to [0, 1]
    img = Image.open(image_path).convert('L')
    img = img.resize((img_size, img_size))
    f_img = np.array(img) / 255.0

## Define the Constant Kernel
* We will assume $k$ is known and seek to optimize $u$.
* A sigma term is included for the blurriness of my image.

In [5]:
kernel_size = 9
sigma_value = 1.5
kernel = get_gaussian_kernel(size=kernel_size, sigma=sigma_value)
K_sparse = create_sparse_blur_matrix(f_img.shape, kernel)

# Flatten the image for matrix operations
f_vec = f_img.flatten()

## Formulate the Problem and Solve with CVXPY

In [ ]:
print(f"Setting up optimization for {image_path}...")

# Variable to solve for (The Sharp Image)
U = cp.Variable((img_size, img_size))
U_vec = cp.vec(U)

# TV Regularization Weight
# INCREASE this if the output is too grainy
# DECREASE this if the output looks too "smeared" or blocky
LAMBDA_TV = 0.05 

# Objective: Minimize ||KU - f||^2 + λ||∇U||_1
# Note: We removed the Tikhonov (L2) term as requested
obj = cp.Minimize(cp.sum_squares(K_sparse @ U_vec - f_vec) + LAMBDA_TV * cp.tv(U))

# Constraints: Keep pixel values within valid range
constraints = [U >= 0, U <= 1]

prob = cp.Problem(obj, constraints)

# Using SCS solver which is robust for TV problems
print("Solving... (This may take ~30-60 seconds)")
prob.solve(solver=cp.SCS, verbose=True)

u_recovered = U.value
u_recovered = u_recovered.T

/tmp/ipykernel_921/479191896.py:5: FutureWarning: 
    You didn't specify the order of the vec expression. The default order
    used in CVXPY is Fortran ('F') order. This default will change to match NumPy's
    default order ('C') in a future version of CVXPY.
    To suppress this warning, please specify the order explicitly.
    
  U_vec = cp.vec(U)
(CVXPY) Apr 19 04:56:44 AM: Your problem has 16384 variables, 32768 constraints, and 0 parameters.
(CVXPY) Apr 19 04:56:44 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Apr 19 04:56:44 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Apr 19 04:56:44 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Apr 19 04:56:44 AM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Apr 19 04:56:44 AM: Compiling problem (target solver=SCS).
(CVXPY) Apr 19 04:56:44 AM: Reduction chain

Setting up optimization for kitten.jpg...
Solving... (This may take ~30-60 seconds)
                                     CVXPY                                     
                                     v1.8.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Apr 19 04:56:44 AM: Applying reduction SCS
(CVXPY) Apr 19 04:56:45 AM: Finished problem compilation (took 1.045e+00 seconds).
(CVXPY) Apr 19 04:56:45 AM: Invoking solver SCS  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------
------------------------------------------------------------------
	       SCS v3.2.11 - Splitting Conic Solver
	(c) Brendan O'Donoghue, Stanford University, 2012
------------------------------------------------------------------
problem:  variables n: 48897, constraints m: 97539
cones: 	  z: primal zero / dual free vars: 16384
	  l: linear vars: 32768
	  q: soc vars: 48387, qsize: 16129
settings: eps_abs: 1.0e-05, eps_rel: 1.0e-05, eps_infeas: 1.0e-07
	  alpha: 1.50, scale: 1.00e-01, adaptive_scale: 1
	  max_iters: 100000, normalize: 1, rho_x: 1.00e-06
	  acceleration_lookback: 10, acceleration_interval: 10
lin-sys:  sparse-direct-amd-qdldl
	  nnz(A): 1411221, nnz(P): 16384
------------------------------------------------------------------
 iter | p

## Visualize the Results

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(f_img, cmap='gray')
plt.title("Your Input Image (Blurry/Original)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(u_recovered, cmap='gray')
plt.title(f"TV Restored (λ={LAMBDA_TV})")
plt.axis('off')

plt.tight_layout()
plt.show()